# Pharmacy Medicine Demand & Expiry-Risk Forecasting

This notebook explores the included local, synthetic pharmacy dataset. It regenerates data, runs a chronological holdout evaluation, forecasts demand, and summarizes lot expiry exposure. No real patient or pharmacy records are used.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from src.generate_data import build_dataset
from src.pipeline import run_pipeline

ROOT = Path.cwd()
build_dataset()
summary = run_pipeline()
summary

## Evaluation

The last 12 weeks per SKU are kept out of training. Recursive predictions are compared with a baseline that repeats the previous four-week average. WAPE is the sum of absolute errors divided by total actual units, so larger-volume products have more weight in the aggregate.

In [ ]:
metrics = pd.read_csv(ROOT / 'evidence/holdout_metrics_by_sku.csv')
metrics.describe().round(3)

In [ ]:
forecasts = pd.read_csv(ROOT / 'evidence/12_week_demand_forecast.csv')
expiry = pd.read_csv(ROOT / 'evidence/expiry_risk_and_reorder.csv')
print('Forecast rows:', len(forecasts))
print('Total units expiring unconsumed in horizon:', int(expiry.units_expiring_unconsumed_12w.sum()))
expiry.sort_values('risk_value_estimate', ascending=False).head(10)

## Interpretation and limits

The reported backtest gain over the simple baseline is modest and applies only to this generated sample. The promotion flag is not used as a feature. Expiry risk assumes the lot snapshot is correct and applies FEFO consumption without replenishment during the 12-week projection. This prototype is for learning and must not be used directly for live purchasing or clinical decisions.